In [2]:
import pandas as pd

In [5]:
df = pd.read_csv("../recipes.csv")

print("shape:", df.shape)
display(df.head(3))

shape: (554161, 4)


,name,cleaned_ingredients,ingredients_ratio,cuisine
0,Worlds Best Mac and Cheese,"[['penne', 170.0], ['cheddar cheese', 28.0], [...","[['penne', 112.137], ['cheddar cheese', 18.47]...",Italian
1,Dilly Macaroni Salad Recipe,"[['macaroni', 140.0], ['cheese', 113.0], ['cel...","[['macaroni', 237.893], ['cheese', 192.014], [...",American
2,Gazpacho,"[['tomato', 1600.0], ['salt', 3.0], ['onion', ...","[['tomato', 651.466], ['salt', 1.221], ['onion...",French


In [ ]:
import pandas as pd
import re
import ast
from collections import defaultdict
from pathlib import Path

# =============================
# Config
# =============================
FLAVORDB_PATH = Path("../flavordb_alias_in_vocab_only.csv")
RECIPE_PATH   = Path("../recipe_cleaned_with_ratio.csv")

OUT_EDGES     = Path("../recipe_molecule_edges.csv")
OUT_RECIPE_SUM= Path("../recipe_molecule_recipe_strength.csv")

CHUNKSIZE = 50_000          # 55만이면 5만~10만 권장 (메모리/속도 보고 조절)
ALPHA = 1.0                 # |M(i)|^alpha 보정 (1.0 추천)
TOPK_PER_RECIPE = None      # 예: 200 넣으면 recipe당 상위 200 molecule만 저장 (용량 줄이기)

# =============================
# Normalization (same style)
# =============================
def norm(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"[_\-]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s']", "", s)
    return s.strip()

# =============================
# Parse molecules field like "{27457, 31252, ...}"
# =============================
_mol_num_re = re.compile(r"\d+")

def parse_molecules(cell) -> list[int]:
    if pd.isna(cell):
        return []
    s = str(cell)
    nums = _mol_num_re.findall(s)
    return [int(x) for x in nums]

# =============================
# Parse ingredients_ratio cell like:
#   "[['penne', 112.137], ['cheddar cheese', 18.47], ...]"
# or possibly already list-like
# =============================
def parse_ingredients_ratio(cell):
    if pd.isna(cell):
        return []
    if isinstance(cell, list):
        return cell
    s = str(cell).strip()
    if not s:
        return []
    try:
        obj = ast.literal_eval(s)
        # 기대: [[name, ratio], ...]
        return obj if isinstance(obj, list) else []
    except Exception:
        return []

# =============================
# 1) Build ingredient -> molecules map
# =============================
flav = pd.read_csv(FLAVORDB_PATH)

if "alias" not in flav.columns:
    raise ValueError(f"'alias' column not found in {FLAVORDB_PATH}. cols={list(flav.columns)}")
if "molecules" not in flav.columns:
    raise ValueError(f"'molecules' column not found in {FLAVORDB_PATH}. cols={list(flav.columns)}")

flav["alias_norm"] = flav["alias"].map(norm)
flav["mol_list"] = flav["molecules"].map(parse_molecules)

# alias_norm 중복이 있으면 molecules 합집합으로 처리 (안전)
ing2mols: dict[str, set[int]] = defaultdict(set)
for a, mols in zip(flav["alias_norm"], flav["mol_list"]):
    if not a:
        continue
    for m in mols:
        ing2mols[a].add(m)

# set -> sorted list로 고정 (반복 시 빠름)
ing2mols = {k: sorted(v) for k, v in ing2mols.items()}
ing2mol_count = {k: len(v) for k, v in ing2mols.items()}

print(f"[INFO] #ingredients with molecules: {len(ing2mols)}")

# =============================
# 2) Stream recipes and write recipe-molecule edges
# =============================
# output 초기화
if OUT_EDGES.exists():
    OUT_EDGES.unlink()
if OUT_RECIPE_SUM.exists():
    OUT_RECIPE_SUM.unlink()

# 레시피 파일 컬럼 체크
tmp_head = pd.read_csv(RECIPE_PATH, nrows=5)
need_cols = ["name", "ingredients_ratio"]
for c in need_cols:
    if c not in tmp_head.columns:
        raise ValueError(f"'{c}' column not found in {RECIPE_PATH}. cols={list(tmp_head.columns)}")

edge_header_written = False
sum_header_written = False

recipe_id_base = 0  # chunk마다 이어붙이기 위한 recipe_id

for chunk_idx, rec in enumerate(pd.read_csv(RECIPE_PATH, chunksize=CHUNKSIZE)):
    # 파싱
    rec["parsed"] = rec["ingredients_ratio"].map(parse_ingredients_ratio)

    edge_rows = []
    sum_rows = []

    for j, row in rec.iterrows():
        rid = recipe_id_base + (j - rec.index[0])  # chunk 내부 연속 id
        rname = row["name"]
        items = row["parsed"]

        # molecule score accumulator for this recipe
        mol_score = defaultdict(float)

        for it in items:
            # 기대 형태: [ingredient_name, ratio]
            if not isinstance(it, (list, tuple)) or len(it) < 1:
                continue

            ing_name = it[0]
            w = 1.0
            if len(it) >= 2:
                try:
                    w = float(it[1])
                except Exception:
                    w = 1.0

            key = norm(ing_name)
            mols = ing2mols.get(key)
            if not mols:
                continue

            denom = (ing2mol_count[key] ** ALPHA) if ing2mol_count[key] > 0 else 1.0
            contrib = w / denom

            for m in mols:
                mol_score[m] += contrib

        if not mol_score:
            sum_rows.append((rid, rname, 0.0, 0))
            continue

        # recipe strength (총 score)
        total = float(sum(mol_score.values()))
        sum_rows.append((rid, rname, total, len(mol_score)))

        # TopK 저장 옵션
        if TOPK_PER_RECIPE is not None and len(mol_score) > TOPK_PER_RECIPE:
            top_items = sorted(mol_score.items(), key=lambda x: x[1], reverse=True)[:TOPK_PER_RECIPE]
        else:
            top_items = mol_score.items()

        for m, s in top_items:
            edge_rows.append((rid, rname, int(m), float(s)))

    # write edges
    edge_df = pd.DataFrame(edge_rows, columns=["recipe_id", "name", "molecule", "score"])
    edge_df.to_csv(OUT_EDGES, mode="a", index=False, header=not edge_header_written, encoding="utf-8")
    edge_header_written = True

    # write recipe summary
    sum_df = pd.DataFrame(sum_rows, columns=["recipe_id", "name", "total_score", "num_molecules"])
    sum_df.to_csv(OUT_RECIPE_SUM, mode="a", index=False, header=not sum_header_written, encoding="utf-8")
    sum_header_written = True

    recipe_id_base += len(rec)

    print(f"[INFO] chunk {chunk_idx} done: recipes={len(rec):,}, edges_written={len(edge_df):,}")

print(f"[DONE] Saved edges to: {OUT_EDGES}")
print(f"[DONE] Saved recipe summary to: {OUT_RECIPE_SUM}")
